# 25 - Teacher Custom Data Single Notebook Demo

This notebook is the single-file custom-data demo for the instructor. It ingests custom `.txt`, `.csv`, or `.jsonl` files, builds a RAG index with the selected Qwen3 embedding model, optionally evaluates retrieval metrics on an instructor benchmark, and finally opens an optional Gradio UI for asking questions over the custom document collection.

In [ ]:
!pip install -q -U gradio "sentence-transformers>=3.0.0" transformers accelerate bitsandbytes peft faiss-cpu rank-bm25 pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
for path in [DRIVE_ROOT, DRIVE_ROOT / 'src']:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Put instructor-provided files here.
CUSTOM_INPUT_DIR = DRIVE_ROOT / 'data/custom_docs'

# Normalized custom corpus and index outputs.
CUSTOM_OUTPUT_CSV = DRIVE_ROOT / 'data/processed/custom_corpus_teacher_demo.csv'
CUSTOM_OUTPUT_JSONL = DRIVE_ROOT / 'data/processed/custom_corpus_teacher_demo.jsonl'
CUSTOM_REPORT_JSON = DRIVE_ROOT / 'reports/custom_ingestion_report_teacher_demo.json'
CUSTOM_INDEX_ROOT = DRIVE_ROOT / 'indexes/custom_teacher_demo'
CUSTOM_BENCHMARK_CSV = DRIVE_ROOT / 'data/custom_benchmark/custom_benchmark.csv'

# Use the same strong embedding family as the final system.
# If Colab memory is limited, replace with 'BAAI/bge-m3'.
EMBEDDING_MODEL = 'Qwen/Qwen3-Embedding-8B'

CUSTOM_INPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device)
print('Project root exists:', DRIVE_ROOT.exists())
print('Source dir exists:', (DRIVE_ROOT / 'src').exists())
print('Put instructor files under:', CUSTOM_INPUT_DIR)
print('Custom index will be written to:', CUSTOM_INDEX_ROOT)
print('Optional custom benchmark path:', CUSTOM_BENCHMARK_CSV)

## 1. Check Custom Documents

Upload the instructor document collection into `data/custom_docs/`. Supported formats: `.txt`, `.csv`, `.jsonl`.

If the folder is empty, this notebook creates a tiny sample file only so that the demo can be tested end-to-end.

In [ ]:
SUPPORTED_SUFFIXES = {'.txt', '.csv', '.jsonl'}
custom_files = [p for p in CUSTOM_INPUT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES]

if not custom_files:
    sample_path = CUSTOM_INPUT_DIR / 'sample_teacher_custom_document.txt'
    sample_path.write_text(
        'Ozel demo dokumani: kira uyusmazliklari ve odeme borcu.\n\n'
        'Kiraci kira bedelini odemezse, ilgili sozlesme ve mevzuat hukumleri kapsaminda yazili bildirim, odeme suresi ve fesih sonucuyla karsilasabilir.\n\n'
        'Bu dosya yalnizca custom RAG pipeline test dosyasidir.\n',
        encoding='utf-8',
    )
    custom_files = [sample_path]
    print('Sample file created:', sample_path)
else:
    print('Custom files found:')
    for path in sorted(custom_files):
        print('-', path)

## 2. Ingest and Normalize Custom Documents

In [ ]:
from src.ingest_custom_documents import ingest_custom_documents

ingestion_report = ingest_custom_documents(
    input_dir=CUSTOM_INPUT_DIR,
    output_csv=CUSTOM_OUTPUT_CSV,
    output_jsonl=CUSTOM_OUTPUT_JSONL,
    report_json=CUSTOM_REPORT_JSON,
    max_chars=1800,
    overlap_chars=180,
    min_chars=30,
)
ingestion_report

In [ ]:
import pandas as pd

custom_df = pd.read_csv(CUSTOM_OUTPUT_CSV, dtype=str, keep_default_na=False)
print('Custom corpus shape:', custom_df.shape)
custom_df[['record_id', 'doc_key', 'article_key', 'citation_label', 'retrieval_text']].head(10)

## 3. Build Custom Dense/BM25 Index

In [ ]:
from src.build_index import build_indexes

index_manifest = build_indexes(
    corpus_path=CUSTOM_OUTPUT_CSV,
    index_root=CUSTOM_INDEX_ROOT,
    embedding_model=EMBEDDING_MODEL,
    text_field='retrieval_text',
    batch_size=8,
    device=device,
    build_dense=True,
    build_bm25=True,
)
index_manifest

In [ ]:
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

print('Custom index exists:', CUSTOM_INDEX_ROOT.exists())
print('Manifest exists:', (CUSTOM_INDEX_ROOT / 'index_manifest.json').exists())

## 4. Retrieval Smoke Test

In [ ]:
from src.retrieval import RetrievalEngine

engine = RetrievalEngine(CUSTOM_INDEX_ROOT, device=device)
smoke_query = 'Bu dokumanlarda hangi hukuki konu anlatiliyor?'
smoke_results = engine.dense_search(smoke_query, top_k=5)
[(row.get('citation_label'), round(float(row.get('score', 0.0)), 4)) for row in smoke_results]

## 5. Optional Custom Benchmark Metrics

If the instructor provides a benchmark CSV, put it at `data/custom_benchmark/custom_benchmark.csv`.

Expected minimum columns:

- `question` or `query`
- one relevant-document column such as `gold_article_keys`, `gold_doc_keys`, `relevant_article_keys`, `relevant_doc_keys`, or `relevant_documents`

The cell below computes retrieval hit/recall, MRR, and nDCG over the custom index. This is the metric-first teacher demo path. The UI is only optional and comes after this section.

In [ ]:
import gc
import json
import math
import re
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.retrieval import RetrievalEngine
from src.reranking import CrossEncoderReranker

RERANKER_MODEL = 'Qwen/Qwen3-Reranker-8B'
USE_RERANKER_FOR_CUSTOM_METRICS = True
CUSTOM_METRICS_TOP_K = 10
CUSTOM_METRICS_CANDIDATE_K = 30


def resolve_custom_benchmark_csv():
    benchmark_dir = DRIVE_ROOT / 'data/custom_benchmark'
    preferred = benchmark_dir / 'custom_benchmark.csv'

    if preferred.exists():
        return preferred

    csv_files = sorted(path for path in benchmark_dir.glob('*.csv') if path.is_file())

    if len(csv_files) == 1:
        print('custom_benchmark.csv bulunamadi; klasordeki tek CSV kullaniliyor:', csv_files[0])
        return csv_files[0]

    if len(csv_files) > 1:
        print('custom_benchmark.csv bulunamadi; birden fazla CSV var. Ilki kullaniliyor:', csv_files[0])
        for path in csv_files:
            print('-', path)
        return csv_files[0]

    return preferred


CUSTOM_BENCHMARK_CSV = resolve_custom_benchmark_csv()


def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('VRAM free/total:', torch.cuda.mem_get_info())


def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def split_keys(value):
    text = str(value or '').strip()
    if not text:
        return set()
    parts = []
    for chunk in text.replace('|', ';').replace(',', ';').split(';'):
        item = chunk.strip()
        if item:
            parts.append(item)
    return set(parts)


def norm_key(value):
    value = str(value or '').lower().strip()
    value = value.translate(str.maketrans('????????', 'cgiosuii'))
    value = re.sub(r'[^a-z0-9]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return value


def retrieved_identity_set(row):
    values = set()
    for key in [
        'article_key',
        'doc_key',
        'record_id',
        'citation_label',
        'law_name_norm',
        'law_name_raw',
        'doc_title',
        'article_no_norm',
        'article_no_raw',
        'article_body',
        'retrieval_text',
        'generation_text',
    ]:
        value = str(row.get(key, '')).strip()
        if value:
            values.add(value)
            values.add(norm_key(value))
    return values


def benchmark_gold_identity_set(item):
    values = set()

    direct_cols = [
        'gold_article_keys',
        'relevant_article_keys',
        'gold_doc_keys',
        'relevant_doc_keys',
        'relevant_documents',
        'gold_documents',
        'article_key',
        'doc_key',
    ]

    for col in direct_cols:
        if col in item.index:
            for value in split_keys(item.get(col, '')):
                values.add(value)
                values.add(norm_key(value))

    source_cols = ['gold_source_canonical', 'gold_source', 'source_url_canonical', 'source_url']
    article_cols = ['gold_article_normalized', 'gold_article', 'gold_article_raw']

    sources = []
    articles = []

    for col in source_cols:
        if col in item.index:
            value = str(item.get(col, '')).strip()
            if value:
                sources.append(value)
                values.add(value)
                values.add(norm_key(value))

    for col in article_cols:
        if col in item.index:
            value = str(item.get(col, '')).strip()
            if value:
                articles.append(value)
                values.add(value)
                values.add(norm_key(value))

    for source in sources:
        for article in articles:
            pair_variants = [
                f'{source} {article}',
                f'{source}_{article}',
                f'{source} m.{article}',
                f'{source} madde {article}',
            ]
            for pair in pair_variants:
                values.add(pair)
                values.add(norm_key(pair))

    return values, sources, articles


def is_relevant_result(result, gold_values, gold_sources, gold_articles):
    result_values = retrieved_identity_set(result)

    if result_values & gold_values:
        return 1

    result_text = ' '.join(str(result.get(k, '')) for k in [
        'article_key',
        'doc_key',
        'citation_label',
        'law_name_norm',
        'law_name_raw',
        'doc_title',
        'article_no_norm',
        'article_no_raw',
        'retrieval_text',
        'generation_text',
    ])
    result_norm = norm_key(result_text)

    source_match = True
    article_match = True

    if gold_sources:
        source_match = any(norm_key(source) and norm_key(source) in result_norm for source in gold_sources)

    if gold_articles:
        article_match = any(norm_key(article) and norm_key(article) in result_norm for article in gold_articles)

    return 1 if source_match and article_match else 0


def dcg_at_k(relevances, k):
    total = 0.0
    for idx, rel in enumerate(relevances[:k], start=1):
        if rel:
            total += 1.0 / math.log2(idx + 1)
    return total


def ndcg_at_k(relevances, k):
    ideal_count = min(sum(1 for item in relevances if item), k)
    if ideal_count == 0:
        return 0.0
    ideal = dcg_at_k([1] * ideal_count, k)
    return dcg_at_k(relevances, k) / ideal if ideal else 0.0


def evaluate_custom_retrieval_metrics(benchmark_csv, index_root):
    benchmark = pd.read_csv(benchmark_csv, dtype=str, keep_default_na=False)

    question_col = first_existing_column(benchmark, ['question', 'query', 'soru'])
    if not question_col:
        raise ValueError(f'Benchmark must include question/query/soru. Columns: {list(benchmark.columns)}')

    print('Pass 1/2: Dense retrieval candidates are being generated...')
    engine = RetrievalEngine(index_root, device=device)

    dense_rows = []
    for _, item in tqdm(benchmark.iterrows(), total=len(benchmark), desc='Dense retrieval'):
        question = str(item.get(question_col, '')).strip()
        gold_values, gold_sources, gold_articles = benchmark_gold_identity_set(item)

        candidates = engine.dense_search(
            question,
            top_k=CUSTOM_METRICS_CANDIDATE_K if USE_RERANKER_FOR_CUSTOM_METRICS else CUSTOM_METRICS_TOP_K,
        )

        dense_rows.append(
            {
                'question': question,
                'gold_values': gold_values,
                'gold_sources': gold_sources,
                'gold_articles': gold_articles,
                'candidates': candidates,
            }
        )

    del engine
    free_gpu_memory()

    reranker = None
    if USE_RERANKER_FOR_CUSTOM_METRICS:
        print('Pass 2/2: Reranking with Qwen3-Reranker-8B...')
        reranker = CrossEncoderReranker(RERANKER_MODEL, device=device)
    else:
        print('Pass 2/2: Reranker disabled; evaluating dense candidates directly...')

    rows = []
    for item in tqdm(dense_rows, desc='Reranking custom benchmark'):
        question = item['question']
        candidates = item['candidates']

        if reranker:
            candidates = reranker.rerank(
                question,
                candidates,
                top_k=CUSTOM_METRICS_TOP_K,
                batch_size=1,
            )

        candidates = candidates[:CUSTOM_METRICS_TOP_K]

        relevances = []
        retrieved_keys = []

        for result in candidates:
            retrieved_keys.append(
                result.get('article_key')
                or result.get('doc_key')
                or result.get('record_id')
                or ''
            )
            relevances.append(
                is_relevant_result(
                    result,
                    item['gold_values'],
                    item['gold_sources'],
                    item['gold_articles'],
                )
            )

        first_hit_rank = next(
            (idx for idx, rel in enumerate(relevances, start=1) if rel),
            None,
        )

        rows.append(
            {
                'question': question,
                'gold_sources': ';'.join(item['gold_sources']),
                'gold_articles': ';'.join(item['gold_articles']),
                'retrieved_keys_top10': ';'.join(str(key) for key in retrieved_keys),
                'hit@5': 1.0 if any(relevances[:5]) else 0.0,
                'hit@10': 1.0 if any(relevances[:10]) else 0.0,
                'recall@5': 1.0 if any(relevances[:5]) else 0.0,
                'recall@10': 1.0 if any(relevances[:10]) else 0.0,
                'mrr': 1.0 / first_hit_rank if first_hit_rank else 0.0,
                'ndcg@5': ndcg_at_k(relevances, 5),
                'ndcg@10': ndcg_at_k(relevances, 10),
            }
        )

    if reranker is not None:
        del reranker

    del dense_rows
    free_gpu_memory()

    result_df = pd.DataFrame(rows)

    summary = {
        'benchmark_csv': str(benchmark_csv),
        'index_root': str(index_root),
        'embedding_model': EMBEDDING_MODEL,
        'reranker_model': RERANKER_MODEL if USE_RERANKER_FOR_CUSTOM_METRICS else None,
        'question_count': int(len(result_df)),
        'metrics': {
            col: float(result_df[col].mean())
            for col in ['hit@5', 'hit@10', 'recall@5', 'recall@10', 'mrr', 'ndcg@5', 'ndcg@10']
        },
    }

    metrics_csv = DRIVE_ROOT / 'outputs/retrieval_eval/custom_teacher_demo_retrieval_metrics.csv'
    summary_json = DRIVE_ROOT / 'outputs/retrieval_eval/custom_teacher_demo_retrieval_summary.json'

    metrics_csv.parent.mkdir(parents=True, exist_ok=True)
    result_df.to_csv(metrics_csv, index=False, encoding='utf-8-sig')
    summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    return summary, result_df


if CUSTOM_BENCHMARK_CSV.exists():
    print('Using custom benchmark:', CUSTOM_BENCHMARK_CSV)
    custom_summary, custom_metric_rows = evaluate_custom_retrieval_metrics(
        CUSTOM_BENCHMARK_CSV,
        CUSTOM_INDEX_ROOT,
    )
    print(json.dumps(custom_summary, ensure_ascii=False, indent=2))
    display(custom_metric_rows.head(20))
else:
    print('Custom benchmark not found:', CUSTOM_BENCHMARK_CSV)
    print('Metric section skipped. Add a CSV under data/custom_benchmark/.')


## 6. Custom QA / Citation / Hallucination Metrics

If the custom benchmark includes a `gold_answer` column, this section generates answers and computes QA/citation/hallucination proxy metrics in the same notebook run.

If the benchmark has no answer column, retrieval metrics are still computed above and this section reports why QA metrics cannot be calculated.

In [ ]:
import gc
import json
import re
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.generation import generate_text, load_llm
from src.prompting import build_rag_prompt
from src.retrieval import RetrievalEngine

CUSTOM_QA_EVAL_LIMIT = None  # set to 10 for a quick smoke test
CUSTOM_QA_TOP_K = 5
CUSTOM_QA_LLM = 'Qwen/Qwen3-32B'


def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('VRAM free/total:', torch.cuda.mem_get_info())


def normalize_answer_text(text):
    text = str(text or '').lower()
    text = re.sub(r'[^0-9a-z??????\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def token_f1_score(prediction, gold):
    pred_tokens = normalize_answer_text(prediction).split()
    gold_tokens = normalize_answer_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = {}
    for token in pred_tokens:
        common[token] = common.get(token, 0) + 1
    overlap = 0
    for token in gold_tokens:
        if common.get(token, 0) > 0:
            overlap += 1
            common[token] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def rouge_l_score(prediction, gold):
    pred = normalize_answer_text(prediction).split()
    ref = normalize_answer_text(gold).split()
    if not pred or not ref:
        return 0.0
    dp = [[0] * (len(ref) + 1) for _ in range(len(pred) + 1)]
    for i, token in enumerate(pred, start=1):
        for j, ref_token in enumerate(ref, start=1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if token == ref_token else max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    precision = lcs / len(pred)
    recall = lcs / len(ref)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def citation_present_in_answer(answer, retrieved):
    answer_text = str(answer or '').lower()
    if 'dayanak' in answer_text or 'kaynak' in answer_text or 'madde' in answer_text:
        return 1.0
    for item in retrieved:
        citation = str(item.get('citation_label', '')).lower().strip()
        if citation and citation in answer_text:
            return 1.0
    return 0.0


def resolve_custom_benchmark_csv():
    benchmark_dir = DRIVE_ROOT / 'data/custom_benchmark'
    preferred = benchmark_dir / 'custom_benchmark.csv'
    if preferred.exists():
        return preferred
    csv_files = sorted(path for path in benchmark_dir.glob('*.csv') if path.is_file())
    if len(csv_files) == 1:
        return csv_files[0]
    if len(csv_files) > 1:
        print('custom_benchmark.csv bulunamadi; birden fazla CSV var. Ilki kullaniliyor:', csv_files[0])
        return csv_files[0]
    return preferred


CUSTOM_BENCHMARK_CSV = resolve_custom_benchmark_csv()


def evaluate_custom_qa_metrics(benchmark_csv, index_root):
    benchmark = pd.read_csv(benchmark_csv, dtype=str, keep_default_na=False)
    question_col = first_existing_column(benchmark, ['question', 'query', 'soru'])
    answer_col = first_existing_column(benchmark, ['gold_answer', 'answer', 'cevap'])
    gold_col = first_existing_column(
        benchmark,
        [
            'gold_article_keys', 'relevant_article_keys', 'gold_doc_keys', 'relevant_doc_keys',
            'relevant_documents', 'gold_documents', 'article_key', 'doc_key',
            'gold_source', 'gold_source_canonical', 'gold_article', 'gold_article_normalized',
        ],
    )
    if not question_col or not answer_col:
        raise ValueError(f'Custom QA eval needs question/query and gold_answer/answer columns. Columns: {list(benchmark.columns)}')
    if CUSTOM_QA_EVAL_LIMIT:
        benchmark = benchmark.head(CUSTOM_QA_EVAL_LIMIT).copy()

    # PASS 1: retrieve contexts, then release embedding model before loading Qwen3-32B.
    print('Pass 1/2: Retrieving contexts for QA benchmark...')
    engine = RetrievalEngine(index_root, device=device)
    qa_inputs = []
    for _, item in tqdm(benchmark.iterrows(), total=len(benchmark), desc='QA retrieval'):
        question = str(item.get(question_col, '')).strip()
        gold_answer = str(item.get(answer_col, '')).strip()
        gold_keys = split_keys(item.get(gold_col, '')) if gold_col else set()
        retrieved = engine.dense_search(question, top_k=CUSTOM_QA_TOP_K)
        retrieval_gold_available = 1.0 if gold_keys and any(retrieved_identity_set(row) & gold_keys for row in retrieved) else 0.0
        qa_inputs.append({
            'question': question,
            'gold_answer': gold_answer,
            'gold_keys': gold_keys,
            'retrieved': retrieved,
            'retrieval_gold_available': retrieval_gold_available,
        })
    del engine
    free_gpu_memory()

    # PASS 2: load LLM only after retrieval engine is released.
    print('Pass 2/2: Generating answers with Qwen3-32B...')
    tokenizer, model = load_llm(CUSTOM_QA_LLM, device=device, load_in_4bit=True)
    rows = []
    try:
        for item in tqdm(qa_inputs, desc='QA generation'):
            question = item['question']
            gold_answer = item['gold_answer']
            retrieved = item['retrieved']
            retrieval_gold_available = item['retrieval_gold_available']
            prompt = build_rag_prompt(question, retrieved, max_context_chars=9000)
            generated = generate_text(
                tokenizer,
                model,
                prompt,
                max_new_tokens=384,
                temperature=0.0,
                top_p=1.0,
                input_max_length=8192,
            )
            citation_present = citation_present_in_answer(generated, retrieved)
            grounded = 1.0 if citation_present and (not item['gold_keys'] or retrieval_gold_available) else 0.0
            rows.append({
                'question': question,
                'gold_answer': gold_answer,
                'generated_answer': generated,
                'exact_match': 1.0 if normalize_answer_text(generated) == normalize_answer_text(gold_answer) else 0.0,
                'token_f1': token_f1_score(generated, gold_answer),
                'rouge_l': rouge_l_score(generated, gold_answer),
                'retrieval_gold_available': retrieval_gold_available,
                'citation_present': citation_present,
                'grounded_citation_score': grounded,
                'unsupported_or_missing_citation': 1.0 - grounded,
            })
    finally:
        del tokenizer
        del model
        del qa_inputs
        free_gpu_memory()

    qa_df = pd.DataFrame(rows)
    metric_cols = [
        'exact_match', 'token_f1', 'rouge_l', 'retrieval_gold_available',
        'citation_present', 'grounded_citation_score', 'unsupported_or_missing_citation',
    ]
    summary = {
        'question_count': int(len(qa_df)),
        'limit': CUSTOM_QA_EVAL_LIMIT,
        'metrics': {col: float(qa_df[col].mean()) for col in metric_cols},
    }
    out_csv = DRIVE_ROOT / 'outputs/generation_eval/custom_teacher_demo_qa_metrics.csv'
    out_json = DRIVE_ROOT / 'outputs/generation_eval/custom_teacher_demo_qa_summary.json'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    qa_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
    out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return summary, qa_df


if CUSTOM_BENCHMARK_CSV.exists():
    try:
        custom_qa_summary, custom_qa_rows = evaluate_custom_qa_metrics(CUSTOM_BENCHMARK_CSV, CUSTOM_INDEX_ROOT)
        print(json.dumps(custom_qa_summary, ensure_ascii=False, indent=2))
        display(custom_qa_rows.head(20))
    except ValueError as exc:
        print('Custom QA/citation/hallucination metrics could not be computed:', exc)
        print('Retrieval metrics above are still valid. Add a gold_answer/answer column to compute QA metrics.')
else:
    print('Custom benchmark not found:', CUSTOM_BENCHMARK_CSV)
    print('Retrieval and QA benchmark metrics require a custom benchmark CSV.')


## 7. One-Page Metric Summary

This cell collects the main retrieval and QA/citation metrics printed above into one compact table for the instructor.


In [ ]:
retrieval_summary_path = DRIVE_ROOT / 'outputs/retrieval_eval/custom_teacher_demo_retrieval_summary.json'
qa_summary_path = DRIVE_ROOT / 'outputs/generation_eval/custom_teacher_demo_qa_summary.json'

summary_rows = []
if retrieval_summary_path.exists():
    retrieval_summary = json.loads(retrieval_summary_path.read_text(encoding='utf-8'))
    for metric, value in retrieval_summary.get('metrics', {}).items():
        summary_rows.append({'section': 'retrieval', 'metric': metric, 'value': value})
else:
    print('Retrieval summary not found yet:', retrieval_summary_path)

if qa_summary_path.exists():
    qa_summary = json.loads(qa_summary_path.read_text(encoding='utf-8'))
    for metric, value in qa_summary.get('metrics', {}).items():
        summary_rows.append({'section': 'qa_citation_grounding', 'metric': metric, 'value': value})
else:
    print('QA summary not found yet. This is normal if the benchmark has no gold_answer column or QA generation was skipped:', qa_summary_path)

if summary_rows:
    metric_summary_df = pd.DataFrame(summary_rows)
    display(metric_summary_df)
else:
    print('No metric summary is available yet. Run the retrieval and QA metric cells above first.')


## 7. Optional Custom Document RAG UI

The teacher-facing metric path is above. This UI is optional for a live qualitative demo. Use retrieval-only mode first. Turn on LLM generation only for selected questions because Qwen3-32B can be slow and memory-heavy.

In [ ]:
from functools import lru_cache
from typing import Any

import gradio as gr

from src.generation import generate_text, load_llm
from src.prompting import build_rag_prompt
from src.retrieval import RetrievalEngine

DEFAULT_LLM = 'Qwen/Qwen3-32B'

@lru_cache(maxsize=1)
def get_custom_engine():
    return RetrievalEngine(CUSTOM_INDEX_ROOT, device=device)

@lru_cache(maxsize=1)
def get_llm(model_name: str, load_in_4bit: bool):
    return load_llm(model_name=model_name, device=device, load_in_4bit=load_in_4bit)

def format_sources(items: list[dict[str, Any]]) -> str:
    lines = []
    for i, item in enumerate(items, start=1):
        citation = item.get('citation_label') or item.get('article_key') or ''
        key = item.get('article_key', '')
        score = item.get('score', '')
        text = str(item.get('generation_text') or item.get('retrieval_text') or '').strip()
        lines.append(f'[{i}] {citation}\nKey: {key} | Score: {score}\n{text}')
    return '\n\n'.join(lines)

def custom_answer(question: str, generate_answer: bool, llm_model: str, top_k: int, max_new_tokens: int):
    question = (question or '').strip()
    if not question:
        return 'Soru bos olamaz.', ''
    if not CUSTOM_INDEX_ROOT.exists():
        return f'Custom index bulunamadi: {CUSTOM_INDEX_ROOT}', ''

    engine = get_custom_engine()
    retrieved = engine.dense_search(question, top_k=int(top_k))
    sources = format_sources(retrieved)
    if not generate_answer:
        return 'Retrieval-only mode. Asagida custom dokumanlardan gelen kaynaklar listelendi.', sources

    get_custom_engine.cache_clear()
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    get_llm.cache_clear()
    tokenizer = None
    model = None
    try:
        tokenizer, model = get_llm(llm_model.strip() or DEFAULT_LLM, True)
    except Exception as exc:
        return f'LLM yuklenemedi. Retrieval-only modda kaynaklar asagida. Hata: {type(exc).__name__}: {exc}', sources

    try:
        prompt = build_rag_prompt(question, retrieved, max_context_chars=9000)
        answer = generate_text(
            tokenizer=tokenizer,
            model=model,
            prompt=prompt,
            max_new_tokens=int(max_new_tokens),
            temperature=0.0,
            top_p=1.0,
            input_max_length=8192,
        )
        return answer, sources
    finally:
        del tokenizer
        del model
        get_llm.cache_clear()
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

with gr.Blocks(title='Teacher Custom Data RAG Demo') as demo:
    gr.Markdown('## Teacher Custom Data RAG Demo')
    gr.Markdown('This UI queries only the custom instructor document collection indexed in this notebook.')
    question_box = gr.Textbox(label='Soru', lines=3, placeholder='Bu dokumanlarda hangi hukuki konu anlatiliyor?')
    with gr.Row():
        generate_box = gr.Checkbox(value=False, label='LLM cevabi uret')
        llm_model_box = gr.Textbox(value=DEFAULT_LLM, label='LLM model')
    with gr.Row():
        top_k_box = gr.Slider(1, 10, value=5, step=1, label='Top-k custom context')
        max_tokens_box = gr.Slider(64, 768, value=384, step=64, label='Max answer tokens')
    run_button = gr.Button('Sor')
    answer_box = gr.Textbox(label='Cevap', lines=12)
    sources_box = gr.Textbox(label='Custom kaynaklar / retrieved chunks', lines=16)
    run_button.click(
        custom_answer,
        inputs=[question_box, generate_box, llm_model_box, top_k_box, max_tokens_box],
        outputs=[answer_box, sources_box],
    )

demo.launch(share=True, debug=True, inline=False)

## Final Outputs

- Custom normalized corpus: `data/processed/custom_corpus_teacher_demo.csv`
- Custom JSONL corpus: `data/processed/custom_corpus_teacher_demo.jsonl`
- Custom ingestion report: `reports/custom_ingestion_report_teacher_demo.json`
- Custom index: `indexes/custom_teacher_demo/`
- Retrieval metrics: `outputs/retrieval_eval/custom_teacher_demo_retrieval_metrics.csv`
- Retrieval summary: `outputs/retrieval_eval/custom_teacher_demo_retrieval_summary.json`
- QA/citation metrics: `outputs/generation_eval/custom_teacher_demo_qa_metrics.csv`
- QA/citation summary: `outputs/generation_eval/custom_teacher_demo_qa_summary.json`
- Optional UI: generated by the final UI cell of this notebook
